# 5. Model Monitoring & CI/CD Pipeline
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Model performance monitoring
- Data drift detection
- SageMaker Model Monitor
- SageMaker Pipeline (CI/CD)
- CloudWatch monitoring

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import sagemaker
import boto3
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'student-performance'
role = sagemaker.get_execution_role()

In [ ]:
model = joblib.load('../models/random_forest_best.joblib')
train_data = pd.read_csv('../data/processed/train.csv')
test_data = pd.read_csv('../data/processed/test.csv')
X_train = train_data.drop(columns=['performance'])
y_train = train_data['performance']
X_test = test_data.drop(columns=['performance'])
y_test = test_data['performance']

## 5.1 Performance Monitoring Over Time

In [ ]:
np.random.seed(42)
n_windows = 10
window_size = len(X_test) // n_windows

monitoring_results = []
for i in range(n_windows):
    indices = np.random.choice(len(X_test), size=window_size, replace=True)
    X_batch = X_test.iloc[indices]
    y_batch = y_test.iloc[indices]
    preds = model.predict(X_batch)
    monitoring_results.append({
        'Window': f'Week {i+1}',
        'Accuracy': accuracy_score(y_batch, preds),
        'F1-Score': f1_score(y_batch, preds),
        'Precision': precision_score(y_batch, preds),
        'Recall': recall_score(y_batch, preds)
    })

monitor_df = pd.DataFrame(monitoring_results)
monitor_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metrics = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for ax, metric, color in zip(axes.flatten(), metrics, colors):
    ax.plot(monitor_df['Window'], monitor_df[metric], marker='o', color=color, linewidth=2, markersize=8)
    ax.axhline(y=monitor_df[metric].mean(), color='gray', linestyle='--', label=f'Mean: {monitor_df[metric].mean():.3f}')
    ax.fill_between(range(n_windows), 
                    monitor_df[metric].mean() - monitor_df[metric].std(),
                    monitor_df[metric].mean() + monitor_df[metric].std(), alpha=0.2, color=color)
    ax.set_title(f'{metric} Over Time')
    ax.legend()
    ax.tick_params(rotation=45)
    ax.set_ylim(0, 1.1)

plt.suptitle('Model Performance Monitoring Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 5.2 Data Drift Detection

In [ ]:
numerical_features = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
                       'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']

drift_results = []
for feature in numerical_features:
    stat, p_value = stats.ks_2samp(X_train[feature], X_test[feature])
    drift_results.append({
        'Feature': feature,
        'KS Statistic': round(stat, 4),
        'P-Value': round(p_value, 4),
        'Drift Detected': p_value < 0.05
    })

drift_df = pd.DataFrame(drift_results)
print('=== Data Drift Detection (KS Test) ===')
drift_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, feature in enumerate(numerical_features[:6]):
    axes[i].hist(X_train[feature], bins=20, alpha=0.5, color='blue', label='Training', density=True)
    axes[i].hist(X_test[feature], bins=20, alpha=0.5, color='red', label='Test', density=True)
    axes[i].set_title(f'{feature}')
    axes[i].legend()
    p_val = drift_df[drift_df['Feature'] == feature]['P-Value'].values[0]
    axes[i].set_xlabel(f'p-value: {p_val}')

plt.suptitle('Data Drift: Training vs Test Distribution', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 5.3 Data Quality Report

In [ ]:
full_data = pd.read_csv('../data/processed/student_processed.csv')

quality_report = pd.DataFrame({
    'Feature': full_data.columns,
    'Missing (%)': (full_data.isnull().sum() / len(full_data) * 100).round(2),
    'Unique Values': full_data.nunique(),
    'Data Type': full_data.dtypes.astype(str)
})
print('=== Data Quality Report ===')
quality_report

## 5.4 Alert System

In [ ]:
ACCURACY_THRESHOLD = 0.70
F1_THRESHOLD = 0.65

print(f'Thresholds — Accuracy: {ACCURACY_THRESHOLD}, F1: {F1_THRESHOLD}\n')

for _, row in monitor_df.iterrows():
    alerts = []
    if row['Accuracy'] < ACCURACY_THRESHOLD:
        alerts.append(f"LOW ACCURACY: {row['Accuracy']:.3f}")
    if row['F1-Score'] < F1_THRESHOLD:
        alerts.append(f"LOW F1: {row['F1-Score']:.3f}")
    status = 'ALERT' if alerts else 'OK'
    msg = ' | '.join(alerts) if alerts else 'All metrics OK'
    print(f"{row['Window']:10s} [{status:5s}] {msg}")

## 5.5 SageMaker Model Monitor — Baseline

In [ ]:
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

model_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    volume_size_in_gb=10,
    max_runtime_in_seconds=1800
)

baseline_data = session.upload_data(
    path='../data/processed/train.csv',
    bucket=bucket,
    key_prefix=f'{prefix}/baseline'
)

model_monitor.suggest_baseline(
    baseline_dataset=baseline_data,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f's3://{bucket}/{prefix}/baseline-results',
    wait=True
)

print('Baseline created successfully')

In [ ]:
# View baseline statistics
baseline_job = model_monitor.latest_baselining_job
print(f'Baseline job: {baseline_job.job_name}')
print(f'Output: s3://{bucket}/{prefix}/baseline-results')

## 5.6 SageMaker Pipeline (CI/CD DAG)

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.parameters import ParameterString
from sagemaker.processing import SKLearnProcessor, ProcessingInput, ProcessingOutput
from sagemaker.sklearn import SKLearn
from sagemaker.inputs import TrainingInput

input_data = ParameterString(name='InputData', default_value=f's3://{bucket}/{prefix}/processed/student_processed.csv')

In [ ]:
# Step 1: Preprocessing
sklearn_processor = SKLearnProcessor(
    framework_version='1.2-1',
    role=role,
    instance_type='ml.m5.large',
    instance_count=1,
    sagemaker_session=session
)

processing_step = ProcessingStep(
    name='DataProcessing',
    processor=sklearn_processor,
    inputs=[ProcessingInput(source=input_data, destination='/opt/ml/processing/input')],
    outputs=[
        ProcessingOutput(output_name='train', source='/opt/ml/processing/output/train'),
        ProcessingOutput(output_name='test', source='/opt/ml/processing/output/test')
    ],
    code='../src/preprocessing.py'
)

In [ ]:
# Step 2: Training
sklearn_estimator = SKLearn(
    entry_point='train.py',
    source_dir='../src/',
    framework_version='1.2-1',
    role=role,
    instance_type='ml.m5.large',
    instance_count=1,
    sagemaker_session=session,
    hyperparameters={'n_estimators': 100, 'max_depth': 10, 'random_state': 42}
)

training_step = TrainingStep(
    name='ModelTraining',
    estimator=sklearn_estimator,
    inputs={'train': TrainingInput(
        s3_data=processing_step.properties.ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri,
        content_type='text/csv'
    )}
)

In [ ]:
# Step 3: Register Model
register_step = RegisterModel(
    name='RegisterModel',
    estimator=sklearn_estimator,
    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.t2.medium'],
    transform_instances=['ml.m5.large'],
    model_package_group_name='StudentPerformanceModelGroup',
    approval_status='Approved'
)

In [ ]:
# Build and run Pipeline
pipeline = Pipeline(
    name='StudentPerformancePipeline',
    parameters=[input_data],
    steps=[processing_step, training_step, register_step],
    sagemaker_session=session
)

pipeline.upsert(role_arn=role)
print(f'Pipeline created: {pipeline.name}')

In [ ]:
# Execute pipeline
execution = pipeline.start()
print(f'Pipeline execution started: {execution.arn}')

execution.wait()
print(f'Status: {execution.describe()["PipelineExecutionStatus"]}')

In [ ]:
# List pipeline steps and status
steps = execution.list_steps()
print('=== Pipeline Steps ===')
for step in steps:
    print(f"{step['StepName']:30s} {step['StepStatus']}")

## Summary
### Monitoring:
- Performance tracked across 10 simulated time windows
- Alert system with accuracy (>0.70) and F1 (>0.65) thresholds
- KS test for data drift on all numerical features
- SageMaker Model Monitor baseline created

### CI/CD:
- SageMaker Pipeline: Data Processing → Training → Model Registration
- Pipeline executed and validated